In [1]:
import pandas as pd
import numpy as np

In [2]:
holdings = pd.read_csv("holdings_5000.csv")
securities = pd.read_csv("security_master_5000.csv")
prices = pd.read_csv("prices_5000.csv")
FX = pd.read_csv("fx.csv")
print(holdings.shape)
print(securities.shape)
print(prices.shape)

(5000, 5)
(5000, 13)
(5000, 7)


In [3]:
#Type Casting & Mandatory Fields
# --------------------------------------
# String type enforcement
string_cols = ["cusip", "ticker", "issuer_id", "issuer_name", "security_type", "day_count_convention", "currency"]
for col in string_cols:
    securities[col] = securities[col].astype(str)

# Decimal / numeric
securities["coupon_rate"] = pd.to_numeric(securities["coupon_rate"], errors="coerce")
securities["par_value"] = pd.to_numeric(securities["par_value"], errors="coerce")
securities["coupon_frequency"] = pd.to_numeric(securities["coupon_frequency"], errors="coerce")

# Date columns
securities["issue_date"] = pd.to_datetime(securities["issue_date"], errors="coerce")
securities["maturity_date"] = pd.to_datetime(securities["maturity_date"], errors="coerce")


In [4]:
mandatory_cols = ["issuer_id", "issuer_name", "security_type", "issue_date", "par_value", "currency","maturity_date"]
for col in mandatory_cols:
  securities[f"flag_null_{col}"] = securities[col].isna()

In [5]:
# Coupon rate 0-25
securities["flag_coupon_rate"] = ~securities["coupon_rate"].between(0, 25)

# Coupon frequency 1,2,4,12
securities["flag_coupon_freq"] = ~securities["coupon_frequency"].isin([1, 2, 4, 12])

# Maturity > Issue date
securities["flag_maturity"] = securities["maturity_date"] <= securities["issue_date"]

# Par value 100 or 1000
securities["flag_par_value"] = ~securities["par_value"].isin([100, 1000])

# Security type
securities["flag_security_type"] = ~securities["security_type"].isin(['BOND','NOTE','TREASURY','CROP'])

# Day count convention
securities["flag_day_count"] = ~securities["day_count_convention"].isin(['30/360','ACT/360','ACT/365'])

# Currency
securities["flag_currency"] = ~securities["currency"].isin(['USD','EUR','INR'])
constraint_cols = [c for c in securities.columns if c.startswith("flag_")]
securities["any_Security_flag"] = securities[constraint_cols].any(axis=1)

In [6]:
Securities_clean_data = securities[securities["any_Security_flag"] == False]
Securities_flagged_data = securities[securities["any_Security_flag"] == True]

In [7]:
pd.set_option('display.max_columns', None)
Securities_clean_data


,isin,cusip,ticker,issuer_id,issuer_name,security_type,coupon_rate,coupon_frequency,issue_date,maturity_date,day_count_convention,par_value,currency,flag_null_issuer_id,flag_null_issuer_name,flag_null_security_type,flag_null_issue_date,flag_null_par_value,flag_null_currency,flag_null_maturity_date,flag_coupon_rate,flag_coupon_freq,flag_maturity,flag_par_value,flag_security_type,flag_day_count,flag_currency,any_Security_flag
0,ISIN100000,CUS100000,JPM,257,Corporate XYZ,BOND,1.32,2,2017-07-24,2027-08-13,30/360,1000,INR,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,ISIN100001,CUS100001,nan,250,Morgan Stanley,BOND,3.76,2,2015-11-16,2025-01-24,30/360,1000,EUR,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,ISIN100002,CUS100002,MS,243,Corporate XYZ,BOND,6.72,2,2015-12-04,2031-12-20,30/360,100,EUR,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,ISIN100003,CUS100003,nan,113,US GOVT,BOND,7.82,1,2014-02-03,2025-07-12,30/360,1000,USD,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,ISIN100004,CUS100004,nan,170,Goldman Sachs,BOND,5.31,1,2014-01-02,2027-05-28,ACT/360,100,USD,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,ISIN104995,CUS104995,JPM,230,Corporate XYZ,BOND,6.99,4,2014-03-07,2025-08-28,ACT/360,100,USD,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4996,ISIN104996,CUS104996,JPM,134,US GOVT,BOND,6.39,1,2014-02-12,2032-08-13,30/360,100,EUR,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4997,ISIN104997,CUS104997,nan,120,Corporate XYZ,BOND,5.32,4,2012-05-26,2028-11-10,30/360,100,INR,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4998,ISIN104998,CUS104998,JPM,163,Morgan Stanley,BOND,7.68,4,2016-05-06,2028-03-14,30/360,100,EUR,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [8]:
print(securities["any_Security_flag"].value_counts())


any_Security_flag
False    4931
True       69
Name: count, dtype: int64


In [9]:
print(Securities_clean_data["any_Security_flag"].value_counts())

any_Security_flag
False    4931
Name: count, dtype: int64


In [10]:
print(Securities_flagged_data["any_Security_flag"].value_counts())

any_Security_flag
True    69
Name: count, dtype: int64


In [11]:
Securities_clean_data[Securities_clean_data["flag_null_maturity_date"]==True]

,isin,cusip,ticker,issuer_id,issuer_name,security_type,coupon_rate,coupon_frequency,issue_date,maturity_date,day_count_convention,par_value,currency,flag_null_issuer_id,flag_null_issuer_name,flag_null_security_type,flag_null_issue_date,flag_null_par_value,flag_null_currency,flag_null_maturity_date,flag_coupon_rate,flag_coupon_freq,flag_maturity,flag_par_value,flag_security_type,flag_day_count,flag_currency,any_Security_flag


In [12]:
Securities_clean_data[Securities_clean_data["flag_null_maturity_date"]==True]

,isin,cusip,ticker,issuer_id,issuer_name,security_type,coupon_rate,coupon_frequency,issue_date,maturity_date,day_count_convention,par_value,currency,flag_null_issuer_id,flag_null_issuer_name,flag_null_security_type,flag_null_issue_date,flag_null_par_value,flag_null_currency,flag_null_maturity_date,flag_coupon_rate,flag_coupon_freq,flag_maturity,flag_par_value,flag_security_type,flag_day_count,flag_currency,any_Security_flag


In [13]:
Securities_clean_data = Securities_clean_data.drop(columns=["flag_null_issuer_id","flag_coupon_rate","flag_null_issuer_name","flag_null_security_type","flag_null_issue_date",
"flag_null_par_value","flag_null_currency","flag_null_maturity_date","flag_coupon_rate","flag_coupon_freq","flag_maturity","flag_par_value","flag_security_type",
"flag_day_count","flag_currency","any_Security_flag"                                                          
])
Securities_clean_data

,isin,cusip,ticker,issuer_id,issuer_name,security_type,coupon_rate,coupon_frequency,issue_date,maturity_date,day_count_convention,par_value,currency
0,ISIN100000,CUS100000,JPM,257,Corporate XYZ,BOND,1.32,2,2017-07-24,2027-08-13,30/360,1000,INR
1,ISIN100001,CUS100001,nan,250,Morgan Stanley,BOND,3.76,2,2015-11-16,2025-01-24,30/360,1000,EUR
2,ISIN100002,CUS100002,MS,243,Corporate XYZ,BOND,6.72,2,2015-12-04,2031-12-20,30/360,100,EUR
3,ISIN100003,CUS100003,nan,113,US GOVT,BOND,7.82,1,2014-02-03,2025-07-12,30/360,1000,USD
4,ISIN100004,CUS100004,nan,170,Goldman Sachs,BOND,5.31,1,2014-01-02,2027-05-28,ACT/360,100,USD
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,ISIN104995,CUS104995,JPM,230,Corporate XYZ,BOND,6.99,4,2014-03-07,2025-08-28,ACT/360,100,USD
4996,ISIN104996,CUS104996,JPM,134,US GOVT,BOND,6.39,1,2014-02-12,2032-08-13,30/360,100,EUR
4997,ISIN104997,CUS104997,nan,120,Corporate XYZ,BOND,5.32,4,2012-05-26,2028-11-10,30/360,100,INR
4998,ISIN104998,CUS104998,JPM,163,Morgan Stanley,BOND,7.68,4,2016-05-06,2028-03-14,30/360,100,EUR


In [14]:
prices["isin"] = prices["isin"].astype(str)
prices["price_date"] = pd.to_datetime(prices["price_date"], errors="coerce")
prices["clean_price"] = pd.to_numeric(prices["clean_price"], errors="coerce")
prices["yield"] = pd.to_numeric(prices["yield"], errors="coerce")
prices["dirty_price"] = pd.to_numeric(prices["dirty_price"], errors="coerce")
prices["currency"] = prices["currency"].astype(str)
prices["price_source"] = prices["price_source"].astype(str)

In [15]:
mandatory_cols1 = ["isin","price_date","clean_price","currency"]
for col in mandatory_cols1:
     prices[f"flag_null_{col}"] = prices[col].isna()

In [16]:
prices["flag_isin_length"] =  prices["isin"].str.len() != 10     
prices["flag_currency_length"] =  prices["currency"].str.len() != 3
prices["pricesflag_clean_price_range"] = ~prices["clean_price"].between(1, 250) 
prices["flag_dirty_vs_clean"] = prices["dirty_price"] <= prices["clean_price"]
prices["flag_yield_range"] = ~prices["yield"].between(-5, 50)
prices["flag_currency_domain"] = ~prices["currency"].isin(["USD", "INR", "EUR"])
constraint_cols = [c for c in prices.columns if c.startswith("flag_")]
prices["any_price_flag"] = prices[constraint_cols].any(axis=1)

In [17]:
pd.set_option('display.max_columns', None)
prices.head()

,isin,price_date,clean_price,dirty_price,yield,currency,price_source,flag_null_isin,flag_null_price_date,flag_null_clean_price,flag_null_currency,flag_isin_length,flag_currency_length,pricesflag_clean_price_range,flag_dirty_vs_clean,flag_yield_range,flag_currency_domain,any_price_flag
0,ISIN101682,2025-06-30,142.11,89.51,11.21,INR,Reuters,False,False,False,False,False,False,False,True,False,False,True
1,ISIN103293,2025-06-30,102.18,68.74,13.46,EUR,Bloomberg,False,False,False,False,False,False,False,True,False,False,True
2,ISIN101446,2025-06-30,147.62,90.68,3.35,INR,Internal,False,False,False,False,False,False,False,True,False,False,True
3,ISIN101519,2025-06-30,94.93,84.75,11.49,USD,Bloomberg,False,False,False,False,False,False,False,True,False,False,True
4,ISIN104310,2025-06-30,105.27,154.47,-0.44,USD,Bloomberg,False,False,False,False,False,False,False,False,False,False,False


In [18]:
prices_clean_data = prices[prices["any_price_flag"] == False]
prices_flagged_data = prices[prices["any_price_flag"] == True]

In [19]:
prices["any_price_flag"].value_counts()

any_price_flag
False    2568
True     2432
Name: count, dtype: int64

In [20]:
prices_clean_data["any_price_flag"].value_counts()

any_price_flag
False    2568
Name: count, dtype: int64

In [21]:
prices_clean_data

,isin,price_date,clean_price,dirty_price,yield,currency,price_source,flag_null_isin,flag_null_price_date,flag_null_clean_price,flag_null_currency,flag_isin_length,flag_currency_length,pricesflag_clean_price_range,flag_dirty_vs_clean,flag_yield_range,flag_currency_domain,any_price_flag
4,ISIN104310,2025-06-30,105.27,154.47,-0.44,USD,Bloomberg,False,False,False,False,False,False,False,False,False,False,False
8,ISIN100111,2025-06-30,78.83,140.03,8.53,INR,Internal,False,False,False,False,False,False,False,False,False,False,False
9,ISIN104477,2025-06-30,59.78,93.82,7.57,EUR,Internal,False,False,False,False,False,False,False,False,False,False,False
10,ISIN103737,2025-06-30,130.55,133.61,-0.52,USD,Internal,False,False,False,False,False,False,False,False,False,False,False
12,ISIN104456,2025-06-30,58.13,147.44,-1.12,INR,Reuters,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4991,ISIN104582,2025-06-30,50.22,115.90,6.97,INR,Reuters,False,False,False,False,False,False,False,False,False,False,False
4992,ISIN100796,2025-06-30,80.50,148.86,1.96,EUR,Bloomberg,False,False,False,False,False,False,False,False,False,False,False
4993,ISIN102534,2025-06-30,70.59,112.97,10.38,INR,Reuters,False,False,False,False,False,False,False,False,False,False,False
4995,ISIN100936,2025-06-30,79.01,100.83,3.80,USD,Bloomberg,False,False,False,False,False,False,False,False,False,False,False


In [22]:
prices_clean_data=prices_clean_data.drop(columns=["flag_null_isin","flag_null_price_date","flag_null_clean_price","flag_null_currency","flag_isin_length","flag_currency_length","pricesflag_clean_price_range","flag_dirty_vs_clean","flag_yield_range","flag_currency_domain","any_price_flag"])

In [23]:
holdings["portfolio_id"] = holdings["portfolio_id"].astype(str)
holdings["as_of_date"] = pd.to_datetime(holdings["as_of_date"], errors="coerce")
holdings["isin"] = holdings["isin"].astype(str)
holdings["quantity"] = pd.to_numeric(holdings["quantity"], errors="coerce")
holdings["book_cost_per_bond"] = pd.to_numeric(holdings["book_cost_per_bond"], errors="coerce")

# Optional NOT NULL (if required logically)
mandatory_cols = ["portfolio_id","as_of_date","isin","quantity","book_cost_per_bond"]
for col in mandatory_cols:
 holdings[f"flag_null_{col}"] = holdings[col].isna()
holdings["flag_invalid_isin_length"] = holdings["isin"].str.len() != 10

flag_cols = [c for c in holdings.columns if c.startswith("flag_")]
holdings["any_schema_flag"] = holdings[flag_cols].any(axis=1)

In [24]:
pd.set_option('display.max_columns', None)
holdings

,portfolio_id,as_of_date,isin,quantity,book_cost_per_bond,flag_null_portfolio_id,flag_null_as_of_date,flag_null_isin,flag_null_quantity,flag_null_book_cost_per_bond,flag_invalid_isin_length,any_schema_flag
0,PF1,2025-06-30,ISIN101487,1,112.85,False,False,False,False,False,False,False
1,PF1,2025-06-30,ISIN100782,375,102.10,False,False,False,False,False,False,False
2,PF2,2025-06-30,ISIN100526,158,105.09,False,False,False,False,False,False,False
3,PF2,2025-06-30,ISIN100000,463,82.63,False,False,False,False,False,False,False
4,PF2,2025-06-30,ISIN100270,127,112.59,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,PF1,2025-06-30,ISIN101060,156,88.12,False,False,False,False,False,False,False
4996,PF2,2025-06-30,ISIN100066,187,80.76,False,False,False,False,False,False,False
4997,PF3,2025-06-30,ISIN102253,141,91.96,False,False,False,False,False,False,False
4998,PF1,2025-06-30,ISIN100340,-23,89.05,False,False,False,False,False,False,False


In [25]:
holdings["any_schema_flag"].value_counts()

any_schema_flag
False    5000
Name: count, dtype: int64

In [26]:
holdings_cleaned_data= holdings[holdings["any_schema_flag"]==False]

In [27]:
holdings_cleaned_data=holdings_cleaned_data.drop(columns=["flag_null_portfolio_id","flag_null_as_of_date","flag_null_isin","flag_null_quantity","flag_null_book_cost_per_bond","flag_invalid_isin_length","any_schema_flag"])

In [28]:
holdings_cleaned_data

,portfolio_id,as_of_date,isin,quantity,book_cost_per_bond
0,PF1,2025-06-30,ISIN101487,1,112.85
1,PF1,2025-06-30,ISIN100782,375,102.10
2,PF2,2025-06-30,ISIN100526,158,105.09
3,PF2,2025-06-30,ISIN100000,463,82.63
4,PF2,2025-06-30,ISIN100270,127,112.59
...,...,...,...,...,...
4995,PF1,2025-06-30,ISIN101060,156,88.12
4996,PF2,2025-06-30,ISIN100066,187,80.76
4997,PF3,2025-06-30,ISIN102253,141,91.96
4998,PF1,2025-06-30,ISIN100340,-23,89.05


In [29]:
FX["date"] = pd.to_datetime(FX["date"], errors="coerce")
FX["currency"] = FX["currency"].astype(str)
FX["usd_rate"] = pd.to_numeric(FX["usd_rate"], errors="coerce")

    # NOT NULL
FX["flag_null_date"] = FX["date"].isna()
FX["flag_null_currency"] = FX["currency"].isna()

    # Currency length
FX["flag_currency_length"] = FX["currency"].str.len() != 3

FX["flag_duplicate_pk"] = FX.duplicated(subset=["date", "currency"], keep=False)
FX["flag_fx_rate_invalid"] = FX["usd_rate"] <= 0
FX["flag_currency_domain"] = ~FX["currency"].isin(["USD", "INR", "EUR"])
constraint_cols = [c for c in FX.columns if c.startswith("flag_")]
FX["any_fx_flag"] = FX[constraint_cols].any(axis=1)

In [30]:
FX

,date,currency,usd_rate,flag_null_date,flag_null_currency,flag_currency_length,flag_duplicate_pk,flag_fx_rate_invalid,flag_currency_domain,any_fx_flag
0,2025-06-30,USD,1.000,False,False,False,False,False,False,False
1,2025-06-30,EUR,1.080,False,False,False,False,False,False,False
2,2025-06-30,INR,0.012,False,False,False,False,False,False,False


In [31]:
FX_clean_data = FX[FX["any_fx_flag"] == False]
FX_flagged_data = FX[FX["any_fx_flag"] == True]

In [32]:
FX_clean_data=FX_clean_data.drop(columns=["flag_null_date","flag_null_currency","flag_currency_length","flag_duplicate_pk","flag_fx_rate_invalid","flag_currency_domain","any_fx_flag"])

In [33]:
prices_clean_data = prices_clean_data.merge(
    Securities_clean_data[["isin", "currency"]],
    on="isin",
    how="left",
    suffixes=("", "_sec")
)

In [34]:
prices_clean_data

,isin,price_date,clean_price,dirty_price,yield,currency,price_source,currency_sec
0,ISIN104310,2025-06-30,105.27,154.47,-0.44,USD,Bloomberg,USD
1,ISIN100111,2025-06-30,78.83,140.03,8.53,INR,Internal,USD
2,ISIN104477,2025-06-30,59.78,93.82,7.57,EUR,Internal,EUR
3,ISIN103737,2025-06-30,130.55,133.61,-0.52,USD,Internal,INR
4,ISIN104456,2025-06-30,58.13,147.44,-1.12,INR,Reuters,INR
...,...,...,...,...,...,...,...,...
2563,ISIN104582,2025-06-30,50.22,115.90,6.97,INR,Reuters,INR
2564,ISIN100796,2025-06-30,80.50,148.86,1.96,EUR,Bloomberg,USD
2565,ISIN102534,2025-06-30,70.59,112.97,10.38,INR,Reuters,EUR
2566,ISIN100936,2025-06-30,79.01,100.83,3.80,USD,Bloomberg,INR


In [35]:
valid_prices = prices_clean_data[
    (prices_clean_data["dirty_price"] >= prices_clean_data["clean_price"]) &
    (prices_clean_data["currency"] == prices_clean_data["currency_sec"])
].copy()

In [36]:
valid_prices

,isin,price_date,clean_price,dirty_price,yield,currency,price_source,currency_sec
0,ISIN104310,2025-06-30,105.27,154.47,-0.44,USD,Bloomberg,USD
2,ISIN104477,2025-06-30,59.78,93.82,7.57,EUR,Internal,EUR
4,ISIN104456,2025-06-30,58.13,147.44,-1.12,INR,Reuters,INR
10,ISIN102952,2025-06-30,86.06,103.61,0.58,USD,Reuters,USD
13,ISIN102200,2025-06-30,53.71,70.54,2.60,EUR,Reuters,EUR
...,...,...,...,...,...,...,...,...
2546,ISIN102298,2025-06-30,111.03,139.80,12.05,INR,Internal,INR
2552,ISIN104335,2025-06-30,67.05,121.69,3.89,INR,Internal,INR
2553,ISIN104496,2025-06-30,135.87,150.34,12.21,USD,Reuters,USD
2554,ISIN103569,2025-06-30,147.32,150.43,3.44,INR,Internal,INR


In [37]:
valid_prices["rn"] = (
    valid_prices
    .sort_values(["isin", "dirty_price"], ascending=[True, False])
    .groupby("isin")
    .cumcount() + 1
)

In [38]:
prices_clean_data=valid_prices[valid_prices["rn"] == 1].copy()

In [39]:
prices_clean_data

,isin,price_date,clean_price,dirty_price,yield,currency,price_source,currency_sec,rn
0,ISIN104310,2025-06-30,105.27,154.47,-0.44,USD,Bloomberg,USD,1
2,ISIN104477,2025-06-30,59.78,93.82,7.57,EUR,Internal,EUR,1
4,ISIN104456,2025-06-30,58.13,147.44,-1.12,INR,Reuters,INR,1
10,ISIN102952,2025-06-30,86.06,103.61,0.58,USD,Reuters,USD,1
13,ISIN102200,2025-06-30,53.71,70.54,2.60,EUR,Reuters,EUR,1
...,...,...,...,...,...,...,...,...,...
2546,ISIN102298,2025-06-30,111.03,139.80,12.05,INR,Internal,INR,1
2552,ISIN104335,2025-06-30,67.05,121.69,3.89,INR,Internal,INR,1
2553,ISIN104496,2025-06-30,135.87,150.34,12.21,USD,Reuters,USD,1
2554,ISIN103569,2025-06-30,147.32,150.43,3.44,INR,Internal,INR,1


In [40]:
prices_flagged_data = prices_flagged_data.copy()

In [41]:
prices_flagged_data["flag_dirty_lt_clean"] = (prices_flagged_data["dirty_price"] < prices_flagged_data["clean_price"])

In [42]:
prices_flagged_data=prices_flagged_data.merge(
    Securities_clean_data[["isin", "currency"]],
    on="isin",
    how="left",
    suffixes=("", "_sec")
)


In [43]:
prices_flagged_data["flag_currency_mismatch"] = (
    prices_flagged_data["currency"] != prices_flagged_data["currency_sec"]
)

In [44]:
prices_flagged_data["flag_isin_not_in_master"] = (prices_flagged_data["currency_sec"].isna())

In [45]:
max_dirty = prices_flagged_data.groupby("isin")["dirty_price"].transform("max")

prices_flagged_data["flag_duplicate_lower_dirty"] = (
    prices_flagged_data["dirty_price"] < max_dirty
)

In [46]:
prices_flagged_data

,isin,price_date,clean_price,dirty_price,yield,currency,price_source,flag_null_isin,flag_null_price_date,flag_null_clean_price,flag_null_currency,flag_isin_length,flag_currency_length,pricesflag_clean_price_range,flag_dirty_vs_clean,flag_yield_range,flag_currency_domain,any_price_flag,flag_dirty_lt_clean,currency_sec,flag_currency_mismatch,flag_isin_not_in_master,flag_duplicate_lower_dirty
0,ISIN101682,2025-06-30,142.11,89.51,11.21,INR,Reuters,False,False,False,False,False,False,False,True,False,False,True,True,INR,False,False,False
1,ISIN103293,2025-06-30,102.18,68.74,13.46,EUR,Bloomberg,False,False,False,False,False,False,False,True,False,False,True,True,EUR,False,False,True
2,ISIN101446,2025-06-30,147.62,90.68,3.35,INR,Internal,False,False,False,False,False,False,False,True,False,False,True,True,INR,False,False,False
3,ISIN101519,2025-06-30,94.93,84.75,11.49,USD,Bloomberg,False,False,False,False,False,False,False,True,False,False,True,True,EUR,True,False,False
4,ISIN102537,2025-06-30,128.93,53.20,2.19,INR,Bloomberg,False,False,False,False,False,False,False,True,False,False,True,True,INR,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2427,ISIN101760,2025-06-30,135.49,54.97,12.77,USD,Bloomberg,False,False,False,False,False,False,False,True,False,False,True,True,EUR,True,False,True
2428,ISIN101191,2025-06-30,111.86,69.13,0.97,USD,Internal,False,False,False,False,False,False,False,True,False,False,True,True,INR,True,False,False
2429,ISIN104558,2025-06-30,103.13,89.29,11.80,USD,Reuters,False,False,False,False,False,False,False,True,False,False,True,True,EUR,True,False,True
2430,ISIN102179,2025-06-30,137.47,108.03,0.25,USD,Internal,False,False,False,False,False,False,False,True,False,False,True,True,INR,True,False,False


In [47]:
group_cols = ["portfolio_id", "as_of_date", "isin"]

holdings_aggregated = (
    holdings_cleaned_data
    .assign(weighted_cost=lambda x: x.quantity * x.book_cost_per_bond)
    .groupby(group_cols, as_index=False)
    .agg(
        quantity=("quantity", "sum"),
        weighted_cost=("weighted_cost", "sum")
    )
)

holdings_aggregated["book_cost_per_bond"] = (
    holdings_aggregated["weighted_cost"]
    .div(holdings_aggregated["quantity"])
    .where(holdings_aggregated["quantity"] != 0)
)

holdings_aggregated.drop(columns="weighted_cost", inplace=True)

In [48]:
holdings_cleaned_data = holdings_aggregated.copy()

# Condition 1: Negative quantity
holdings_cleaned_data["flag_negative_quantity"] = (
    holdings_cleaned_data["quantity"] < 0
)

# Condition 2: Negative book cost
holdings_cleaned_data["flag_negative_book_cost"] = (
    holdings_cleaned_data["book_cost_per_bond"] < 0
)

# Condition 3: ISIN not in security master
holdings_cleaned_data = holdings_cleaned_data.merge(
    Securities_clean_data[["isin"]],
    on="isin",
    how="left",
    indicator=True
)

holdings_cleaned_data["flag_isin_not_in_master"] = (
    holdings_cleaned_data["_merge"] == "left_only"
)

holdings_cleaned_data.drop(columns="_merge", inplace=True)
holdings_cleaned_data = holdings_cleaned_data[
    ~(holdings_cleaned_data["flag_negative_quantity"] |
      holdings_cleaned_data["flag_negative_book_cost"])
]

In [49]:
holdings_cleaned_data=holdings_cleaned_data.drop(columns=["flag_negative_quantity","flag_negative_book_cost","flag_isin_not_in_master"])

In [50]:
holdings_cleaned_data

,portfolio_id,as_of_date,isin,quantity,book_cost_per_bond
1,PF1,2025-06-30,ISIN100006,123,102.015528
2,PF1,2025-06-30,ISIN100008,3,98.290000
3,PF1,2025-06-30,ISIN100015,129,110.540000
4,PF1,2025-06-30,ISIN100016,496,111.380000
5,PF1,2025-06-30,ISIN100020,202,83.790000
...,...,...,...,...,...
4262,PF3,2025-06-30,ISIN104980,296,110.940000
4263,PF3,2025-06-30,ISIN104985,87,99.690000
4264,PF3,2025-06-30,ISIN104991,103,116.080000
4265,PF3,2025-06-30,ISIN104995,411,118.890000


In [51]:
holdings_flagged_data = holdings_aggregated.copy()

In [52]:
holdings_flagged_data["flag_negative_quantity"] = (
   holdings_flagged_data["quantity"] < 0
)

# Condition 2: Negative book cost
holdings_flagged_data["flag_negative_book_cost"] = (
    holdings_flagged_data["book_cost_per_bond"] < 0
)

# Condition 3: ISIN not in security master
holdings_flagged_data = holdings_flagged_data.merge(
    Securities_clean_data[["isin"]],
    on="isin",
    how="left",
    indicator=True
)

holdings_flagged_data["flag_isin_not_in_master"] = (
    holdings_flagged_data["_merge"] == "Right_only"
)

holdings_flagged_data.drop(columns="_merge", inplace=True)
holdings_flagged_data = holdings_flagged_data[
    (holdings_flagged_data["flag_negative_quantity"] |
     holdings_flagged_data["flag_negative_book_cost"])
]

In [53]:
FX_clean_data.loc[FX_clean_data["currency"] == "EUR", "usd_rate"] = 1.1789
FX_clean_data.loc[FX_clean_data["currency"] == "INR", "usd_rate"] = 0.0117

In [54]:
valuation_df = (
    holdings_cleaned_data
    .merge(Securities_clean_data.rename(columns={"currency": "security_currency"}), on="isin", how="inner")
    .merge(prices_clean_data, 
           left_on=["isin", "as_of_date"], 
           right_on=["isin", "price_date"],
           how="inner")
    .merge(FX_clean_data, 
           left_on=["as_of_date", "currency"], 
           right_on=["date", "currency"],
           how="left")
)

In [55]:
valuation_df

,portfolio_id,as_of_date,isin,quantity,book_cost_per_bond,cusip,ticker,issuer_id,issuer_name,security_type,coupon_rate,coupon_frequency,issue_date,maturity_date,day_count_convention,par_value,security_currency,price_date,clean_price,dirty_price,yield,currency,price_source,currency_sec,rn,date,usd_rate
0,PF1,2025-06-30,ISIN100027,700,106.672900,CUS100027,GS,263,Morgan Stanley,BOND,4.55,2,2011-05-25,2025-05-21,30/360,1000,INR,2025-06-30,91.32,134.37,4.70,INR,Reuters,INR,1,2025-06-30,0.0117
1,PF1,2025-06-30,ISIN100034,133,85.660000,CUS100034,MS,199,Morgan Stanley,BOND,0.69,4,2011-09-17,2025-03-07,30/360,100,USD,2025-06-30,64.24,93.39,6.74,USD,Internal,USD,1,2025-06-30,1.0000
2,PF1,2025-06-30,ISIN100061,7,103.440000,CUS100061,JPM,181,Morgan Stanley,BOND,3.46,4,2012-08-10,2032-10-26,ACT/360,100,INR,2025-06-30,112.57,122.41,14.37,INR,Bloomberg,INR,1,2025-06-30,0.0117
3,PF1,2025-06-30,ISIN100104,278,92.020000,CUS100104,T,266,Goldman Sachs,BOND,3.07,2,2018-03-13,2030-09-01,ACT/360,100,EUR,2025-06-30,80.83,91.70,13.19,EUR,Reuters,EUR,1,2025-06-30,1.1789
4,PF1,2025-06-30,ISIN100105,362,86.760000,CUS100105,nan,258,Corporate XYZ,BOND,0.36,2,2017-09-14,2032-11-24,ACT/360,1000,USD,2025-06-30,104.31,126.10,1.99,USD,Internal,USD,1,2025-06-30,1.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
589,PF3,2025-06-30,ISIN104893,221,106.550000,CUS104893,nan,216,US GOVT,BOND,6.12,4,2012-01-07,2025-08-19,ACT/360,1000,EUR,2025-06-30,114.30,117.96,13.43,EUR,Bloomberg,EUR,1,2025-06-30,1.1789
590,PF3,2025-06-30,ISIN104928,221,105.880000,CUS104928,T,237,Goldman Sachs,BOND,9.41,4,2013-10-13,2029-11-30,ACT/360,100,USD,2025-06-30,92.54,123.44,-0.46,USD,Internal,USD,1,2025-06-30,1.0000
591,PF3,2025-06-30,ISIN104957,408,101.836936,CUS104957,GS,175,Goldman Sachs,BOND,1.56,4,2014-07-25,2027-09-11,30/360,100,USD,2025-06-30,58.16,152.48,4.73,USD,Reuters,USD,1,2025-06-30,1.0000
592,PF3,2025-06-30,ISIN104979,371,106.360000,CUS104979,GS,189,Morgan Stanley,BOND,3.69,2,2011-05-04,2026-07-22,ACT/360,1000,INR,2025-06-30,101.96,103.79,3.13,INR,Internal,INR,1,2025-06-30,0.0117


In [56]:
valuation_df["usd_rate"] = valuation_df["usd_rate"].fillna(1.0)

In [57]:
valuation_df["market_value_local"] = (
    valuation_df["quantity"] *
    (valuation_df["clean_price"] / 100.0) *
    valuation_df["par_value"]
)

valuation_df["market_value_usd"] = (
    valuation_df["market_value_local"] *
    valuation_df["usd_rate"]
)

valuation_df["book_value"] = (
    valuation_df["quantity"] *
    valuation_df["book_cost_per_bond"]
)

valuation_df["book_value_usd"] = (
    valuation_df["book_value"] *
    valuation_df["usd_rate"]
)

valuation_df["unrealized_pnl"] = (
    valuation_df["market_value_usd"] -
    valuation_df["book_value_usd"]
)

In [58]:
valuation_result = valuation_df[[
    "portfolio_id",
    "as_of_date",
    "quantity",
    "isin",
    "book_cost_per_bond",
    "issuer_name",
    "security_currency",
    "maturity_date",
    "yield",
    "par_value",
    "clean_price",
    "market_value_local",
    "usd_rate",
    "market_value_usd",
    "book_value",
    "book_value_usd",
    "unrealized_pnl"
]].copy()

In [59]:
valuation_result

,portfolio_id,as_of_date,quantity,isin,book_cost_per_bond,issuer_name,security_currency,maturity_date,yield,par_value,clean_price,market_value_local,usd_rate,market_value_usd,book_value,book_value_usd,unrealized_pnl
0,PF1,2025-06-30,700,ISIN100027,106.672900,Morgan Stanley,INR,2025-05-21,4.70,1000,91.32,639240.00,0.0117,7479.108000,74671.03,873.651051,6605.456949
1,PF1,2025-06-30,133,ISIN100034,85.660000,Morgan Stanley,USD,2025-03-07,6.74,100,64.24,8543.92,1.0000,8543.920000,11392.78,11392.780000,-2848.860000
2,PF1,2025-06-30,7,ISIN100061,103.440000,Morgan Stanley,INR,2032-10-26,14.37,100,112.57,787.99,0.0117,9.219483,724.08,8.471736,0.747747
3,PF1,2025-06-30,278,ISIN100104,92.020000,Goldman Sachs,EUR,2030-09-01,13.19,100,80.83,22470.74,1.1789,26490.755386,25581.56,30158.101084,-3667.345698
4,PF1,2025-06-30,362,ISIN100105,86.760000,Corporate XYZ,USD,2032-11-24,1.99,1000,104.31,377602.20,1.0000,377602.200000,31407.12,31407.120000,346195.080000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
589,PF3,2025-06-30,221,ISIN104893,106.550000,US GOVT,EUR,2025-08-19,13.43,1000,114.30,252603.00,1.1789,297793.676700,23547.55,27760.206695,270033.470005
590,PF3,2025-06-30,221,ISIN104928,105.880000,Goldman Sachs,USD,2029-11-30,-0.46,100,92.54,20451.34,1.0000,20451.340000,23399.48,23399.480000,-2948.140000
591,PF3,2025-06-30,408,ISIN104957,101.836936,Goldman Sachs,USD,2027-09-11,4.73,100,58.16,23729.28,1.0000,23729.280000,41549.47,41549.470000,-17820.190000
592,PF3,2025-06-30,371,ISIN104979,106.360000,Morgan Stanley,INR,2026-07-22,3.13,1000,101.96,378271.60,0.0117,4425.777720,39459.56,461.676852,3964.100868


In [60]:
valuation_result["maturity_days"] = (
    valuation_result["maturity_date"] - valuation_df["as_of_date"]
).dt.days

In [61]:
valuation_result["maturity_status"] = np.where(
    valuation_result["maturity_days"] <= 0,
    "matured",
    "active"
)

In [62]:
valuation_result["maturity_year"] = np.where(
    valuation_result["maturity_days"] < 0,
    0,
    valuation_result["maturity_days"] / 365
)

In [63]:
valuation_result["rate_risk_flag"] = np.where(
    valuation_result["maturity_year"] > 1,
    "rate_sensitive",
    "low_rate_sensitive"
)

In [64]:
valuation_result["fx_risk_type"] = np.where(
    valuation_result["security_currency"] == "USD",
    "NO_FX_Risk",
    "FX_Exposed"
)

In [65]:
portfolio_total_mv_usd = (
    valuation_result.groupby("portfolio_id", as_index=False).agg(Portfolio_mv_usd=("market_value_usd","sum")))

portfolio_total_mv_usd["Portfolio_mv_usd"]=(portfolio_total_mv_usd["Portfolio_mv_usd"].round(2))
portfolio_total_mv_usd

,portfolio_id,Portfolio_mv_usd
0,PF1,15248289.63
1,PF2,14581200.44
2,PF3,17477292.68


In [66]:
currency_exposure = (
    valuation_result
        .groupby("security_currency", as_index=False)
        .agg(currency_mv_usd=("market_value_usd", "sum"))
)

currency_exposure["currency_mv_usd"] = (
    currency_exposure["currency_mv_usd"].round(2)
)
currency_exposure

,security_currency,currency_mv_usd
0,EUR,23525894.52
1,INR,309321.03
2,USD,23471567.20


In [67]:
valuation_result = valuation_result.merge(
    currency_exposure,
    on=["security_currency"],
    how="left"
)

In [68]:
valuation_result = valuation_result.merge(
    portfolio_total_mv_usd,
    on="portfolio_id",
    how="left"
)

In [69]:
valuation_result["fx_concentration_pct"] = (
    valuation_result["currency_mv_usd"] /
    valuation_result["Portfolio_mv_usd"]
) * 100

In [70]:
def classify_fx_risk(pct):
    if pct > 30:
        return "High FX Risk"
    elif pct < 20:
        return "Low FX Risk"
    else:
        return "Moderate FX Risk"

valuation_result["fx_risk_Flag"] = (
    valuation_result["fx_concentration_pct"]
        .apply(classify_fx_risk))

In [71]:
valuation_result["pnl_pct"] = (
    valuation_result["unrealized_pnl"] /
    valuation_result["book_value_usd"]
).where(valuation_result["book_value_usd"] != 0)


In [72]:
valuation_result["fx_stress_5pct"] = np.where(
    valuation_result["security_currency"] != "USD",
    valuation_result["market_value_usd"] * 0.05,
    0
)

valuation_result["fx_stress_10pct"] = np.where(
    valuation_result["security_currency"] != "USD",
    valuation_result["market_value_usd"] * 0.10,
    0
)

In [73]:
valuation_result["risk_market_value"] = np.where(
    valuation_result["maturity_status"] == "active",
    valuation_result["market_value_usd"],
    0
)

In [74]:
valuation_result["risk_quantity"] = np.where(
    valuation_result["maturity_status"] == "active",
    valuation_result["quantity"],
    0
)

In [75]:
valuation_result

,portfolio_id,as_of_date,quantity,isin,book_cost_per_bond,issuer_name,security_currency,maturity_date,yield,par_value,clean_price,market_value_local,usd_rate,market_value_usd,book_value,book_value_usd,unrealized_pnl,maturity_days,maturity_status,maturity_year,rate_risk_flag,fx_risk_type,currency_mv_usd,Portfolio_mv_usd,fx_concentration_pct,fx_risk_Flag,pnl_pct,fx_stress_5pct,fx_stress_10pct,risk_market_value,risk_quantity
0,PF1,2025-06-30,700,ISIN100027,106.672900,Morgan Stanley,INR,2025-05-21,4.70,1000,91.32,639240.00,0.0117,7479.108000,74671.03,873.651051,6605.456949,-40,matured,0.000000,low_rate_sensitive,FX_Exposed,309321.03,15248289.63,2.028562,Low FX Risk,7.560750,373.955400,747.910800,0.000000,0
1,PF1,2025-06-30,133,ISIN100034,85.660000,Morgan Stanley,USD,2025-03-07,6.74,100,64.24,8543.92,1.0000,8543.920000,11392.78,11392.780000,-2848.860000,-115,matured,0.000000,low_rate_sensitive,NO_FX_Risk,23471567.20,15248289.63,153.929180,High FX Risk,-0.250058,0.000000,0.000000,0.000000,0
2,PF1,2025-06-30,7,ISIN100061,103.440000,Morgan Stanley,INR,2032-10-26,14.37,100,112.57,787.99,0.0117,9.219483,724.08,8.471736,0.747747,2675,active,7.328767,rate_sensitive,FX_Exposed,309321.03,15248289.63,2.028562,Low FX Risk,0.088264,0.460974,0.921948,9.219483,7
3,PF1,2025-06-30,278,ISIN100104,92.020000,Goldman Sachs,EUR,2030-09-01,13.19,100,80.83,22470.74,1.1789,26490.755386,25581.56,30158.101084,-3667.345698,1889,active,5.175342,rate_sensitive,FX_Exposed,23525894.52,15248289.63,154.285465,High FX Risk,-0.121604,1324.537769,2649.075539,26490.755386,278
4,PF1,2025-06-30,362,ISIN100105,86.760000,Corporate XYZ,USD,2032-11-24,1.99,1000,104.31,377602.20,1.0000,377602.200000,31407.12,31407.120000,346195.080000,2704,active,7.408219,rate_sensitive,NO_FX_Risk,23471567.20,15248289.63,153.929180,High FX Risk,11.022822,0.000000,0.000000,377602.200000,362
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
589,PF3,2025-06-30,221,ISIN104893,106.550000,US GOVT,EUR,2025-08-19,13.43,1000,114.30,252603.00,1.1789,297793.676700,23547.55,27760.206695,270033.470005,50,active,0.136986,low_rate_sensitive,FX_Exposed,23525894.52,17477292.68,134.608346,High FX Risk,9.727358,14889.683835,29779.367670,297793.676700,221
590,PF3,2025-06-30,221,ISIN104928,105.880000,Goldman Sachs,USD,2029-11-30,-0.46,100,92.54,20451.34,1.0000,20451.340000,23399.48,23399.480000,-2948.140000,1614,active,4.421918,rate_sensitive,NO_FX_Risk,23471567.20,17477292.68,134.297500,High FX Risk,-0.125992,0.000000,0.000000,20451.340000,221
591,PF3,2025-06-30,408,ISIN104957,101.836936,Goldman Sachs,USD,2027-09-11,4.73,100,58.16,23729.28,1.0000,23729.280000,41549.47,41549.470000,-17820.190000,803,active,2.200000,rate_sensitive,NO_FX_Risk,23471567.20,17477292.68,134.297500,High FX Risk,-0.428891,0.000000,0.000000,23729.280000,408
592,PF3,2025-06-30,371,ISIN104979,106.360000,Morgan Stanley,INR,2026-07-22,3.13,1000,101.96,378271.60,0.0117,4425.777720,39459.56,461.676852,3964.100868,387,active,1.060274,rate_sensitive,FX_Exposed,309321.03,17477292.68,1.769845,Low FX Risk,8.586311,221.288886,442.577772,4425.777720,371
